### Load Environment Variables

Load API keys from a `.env` file. This is required to access OpenAI's API for generating embeddings used in the RAG (Retrieval Augmented Generation) system.

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

### Install Required Packages

Install the necessary packages for PDF loading, text splitting, embeddings, and vector storage. Run this cell once to set up dependencies: `pypdf` for PDF parsing, `faiss-cpu` for efficient vector similarity search, and LangChain packages for document processing.

In [2]:
# Install required packages (run once)
# !pip install pypdf faiss-cpu langchain-community langchain-text-splitters

### Import Libraries and Initialize Embeddings

Import all necessary libraries for RAG: PDF loader, text splitter, OpenAI embeddings, and FAISS vector store. Initialize the OpenAI embeddings model that will convert text chunks into vector representations for semantic search.

In [6]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from typing import List, Optional

# Initialize OpenAI embeddings
embeddings = OpenAIEmbeddings()

print("✓ Imports loaded successfully")

✓ Imports loaded successfully


### Define PDFVectorDB Class

Create a comprehensive class that encapsulates the entire RAG pipeline:
- **load_pdf()**: Loads PDF files and splits them into chunks using RecursiveCharacterTextSplitter
- **vectorize_and_store()**: Converts text chunks to embeddings and stores them in FAISS vector database
- **semantic_search()**: Performs similarity search to find relevant document chunks for a query
- **similarity_search()**: Simple wrapper for basic similarity search

This class manages the complete workflow from PDF loading to semantic search.

In [7]:
class PDFVectorDB:
    """
    A class to load PDFs, vectorize them, and perform semantic search.
    """
    
    def __init__(self):
        self.embeddings = OpenAIEmbeddings()
        self.vector_store: Optional[FAISS] = None
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=200,
            length_function=len,
        )
    
    def load_pdf(self, pdf_path: str) -> List[Document]:
        """
        Load a PDF file and split it into chunks.
        
        Args:
            pdf_path: Path to the PDF file
            
        Returns:
            List of Document objects containing the PDF content
        """
        print(f"Loading PDF: {pdf_path}")
        loader = PyPDFLoader(pdf_path)
        documents = loader.load()
        
        # Split documents into chunks
        chunks = self.text_splitter.split_documents(documents)
        print(f"✓ Loaded {len(documents)} pages, split into {len(chunks)} chunks")
        
        return chunks
    
    def vectorize_and_store(self, documents: List[Document]) -> None:
        """
        Vectorize documents and store them in FAISS vector database.
        
        Args:
            documents: List of Document objects to vectorize
        """
        print("Vectorizing documents with OpenAI embeddings...")
        self.vector_store = FAISS.from_documents(documents, self.embeddings)
        print(f"✓ Successfully vectorized and stored {len(documents)} document chunks")
    
    def load_and_vectorize_pdf(self, pdf_path: str) -> None:
        """
        Convenience method to load PDF and vectorize in one step.
        
        Args:
            pdf_path: Path to the PDF file
        """
        documents = self.load_pdf(pdf_path)
        self.vectorize_and_store(documents)
    
    def semantic_search(
        self, 
        query: str, 
        k: int = 4,
        return_source_documents: bool = True
    ) -> dict:
        """
        Perform semantic search on the vector database.
        
        Args:
            query: The search query
            k: Number of results to return (default: 4)
            return_source_documents: Whether to return source documents (default: True)
            
        Returns:
            Dictionary containing:
                - 'results': List of search results with content and metadata
                - 'source_documents': List of source Document objects (if requested)
        """
        if self.vector_store is None:
            raise ValueError("No documents loaded. Please load a PDF first using load_and_vectorize_pdf()")
        
        # Perform similarity search
        results = self.vector_store.similarity_search_with_score(query, k=k)
        
        # Format results
        formatted_results = []
        source_documents = []
        
        for doc, score in results:
            formatted_results.append({
                'content': doc.page_content,
                'metadata': doc.metadata,
                'similarity_score': float(score)
            })
            if return_source_documents:
                source_documents.append(doc)
        
        return {
            'query': query,
            'results': formatted_results,
            'source_documents': source_documents if return_source_documents else None
        }
    
    def similarity_search(self, query: str, k: int = 4) -> List[Document]:
        """
        Simple similarity search that returns Document objects.
        
        Args:
            query: The search query
            k: Number of results to return
            
        Returns:
            List of Document objects
        """
        if self.vector_store is None:
            raise ValueError("No documents loaded. Please load a PDF first using load_and_vectorize_pdf()")
        
        return self.vector_store.similarity_search(query, k=k)

# Create an instance
pdf_db = PDFVectorDB()
print("✓ PDFVectorDB class created")

✓ PDFVectorDB class created


### Load and Vectorize a PDF Document

Load a PDF file, split it into chunks, convert chunks to embeddings using OpenAI, and store them in the FAISS vector database. This prepares the document for semantic search. The text splitter uses a chunk size of 1000 characters with 200 character overlap to preserve context.

In [8]:
# Example usage: Load a PDF and vectorize it
# Replace 'your_document.pdf' with the path to your PDF file

pdf_path = "book.pdf"
pdf_db.load_and_vectorize_pdf(pdf_path)

# Or load and vectorize separately:
# documents = pdf_db.load_pdf(pdf_path)
# pdf_db.vectorize_and_store(documents)

print("Ready to load PDFs! Uncomment the code above and provide a PDF path.")

Loading PDF: book.pdf
✓ Loaded 166 pages, split into 612 chunks
Vectorizing documents with OpenAI embeddings...
✓ Successfully vectorized and stored 612 document chunks
Ready to load PDFs! Uncomment the code above and provide a PDF path.


### Perform Semantic Search

Execute a semantic search query on the vectorized PDF. The search finds the most relevant document chunks based on semantic similarity (not just keyword matching). Results include the content, metadata (like page numbers), and similarity scores showing how closely each chunk matches the query.

In [9]:
# Example: Perform semantic search
# After loading a PDF, you can search like this:

results = pdf_db.semantic_search("What is the main topic?", k=3)

print("Search Results:")
print("=" * 50)
for i, result in enumerate(results['results'], 1):
    print(f"\nResult {i} (Score: {result['similarity_score']:.4f}):")
    print(f"Page: {result['metadata'].get('page', 'N/A')}")
    print(f"Content: {result['content'][:200]}...")
    print("-" * 50)

print("Ready for semantic search! Load a PDF first, then uncomment the code above.")

Search Results:

Result 1 (Score: 0.4671):
Page: 11
Content: 12 
   
 Government can never duplicate the variety and diversity of individual 
action. At any moment in time, by imposing uniform standards in housing, or 
nutrition, or clothing, government could u...
--------------------------------------------------

Result 2 (Score: 0.4682):
Page: 102
Content: American literature and discussion in terms of the meaning attached to the terms 
competition and monopoly in Europe tend to believe that there is a much greater 
degree of monopoly in the United Stat...
--------------------------------------------------

Result 3 (Score: 0.4767):
Page: 6
Content: government activities that has so far been eliminate d and that victory is by no 
means final. In respect of many of the other items, we have moved still farther 
away from the principles espoused in ...
--------------------------------------------------
Ready for semantic search! Load a PDF first, then uncomment the code above.


### Standalone Semantic Search Function

Create a standalone function wrapper for semantic search that can be used independently of the class instance. This provides a simpler function-based interface for performing semantic searches on the PDF vector database.

In [ ]:
# Standalone function for semantic search (if you prefer a function-based approach)
def semantic_search_pdf(query: str, pdf_db_instance: PDFVectorDB = None, k: int = 4) -> dict:
    """
    Standalone function to perform semantic search on the PDF vector database.
    
    Args:
        query: The search query
        pdf_db_instance: PDFVectorDB instance (uses global pdf_db if None)
        k: Number of results to return
        
    Returns:
        Dictionary with search results
    """
    if pdf_db_instance is None:
        pdf_db_instance = pdf_db
    
    return pdf_db_instance.semantic_search(query, k=k)

# Example usage:
# results = semantic_search_pdf("What are the key concepts?", k=5)
# print(results)

print("✓ Standalone semantic_search_pdf() function created")

### Complete RAG Workflow Example

Documentation showing the complete workflow for using the RAG system: loading PDFs, performing semantic searches, and accessing results. This serves as a reference guide for using both the class-based and function-based approaches to RAG.

In [ ]:
# Complete example workflow:
# 
# 1. Load and vectorize a PDF:
#    pdf_db.load_and_vectorize_pdf("document.pdf")
#
# 2. Perform semantic search:
#    results = pdf_db.semantic_search("What is machine learning?", k=3)
#
# 3. Access results:
#    for result in results['results']:
#        print(f"Score: {result['similarity_score']}")
#        print(f"Content: {result['content']}")
#        print(f"Metadata: {result['metadata']}")
#        print("-" * 50)
#
# 4. Or use the standalone function:
#    results = semantic_search_pdf("What is machine learning?", k=3)

print("Example workflow documented above. Uncomment to use!")